In [ ]:
library(destiny)
library(tidyverse)
library(cowplot)

In [ ]:
po <- function(h = 4, w = 5, p = 16) options(repr.plot.height = h, repr.plot.width = w, repr.plot.pointsize = p)

In [ ]:
data(guo_norm)

In [ ]:
dm <- DiffusionMap(guo_norm, distance = 'euclidean')

In [ ]:
po()
plot(dm, 1:2, col_by = 'num_cells', pal = scales::viridis_pal())

In [ ]:
pt <- DPT(dm)

In [ ]:
po()
plot.DPT(pt, root = 2)

In [ ]:
gr <- gene_relevance(dm)

In [ ]:
po()
plot_differential_map(gr, genes = 'Klf4')

In [ ]:
plot_gradient_legend <- function(n_bins = 5L, pal = eval(formals(plot_gene_relevance_rank)$pal)) {
    import::here(.from = ggplot2, scale_x_continuous, scale_y_continuous, scale_fill_gradient, expand_scale, labs, guides)
    import::here(.from = tidyr, crossing)
    import::here(.from = ggplot.multistats  , stat_summaries_hex)

    n_smp <- n_bins*10L
    n_fct <- n_smp^2

    crossing(A = seq_len(n_smp), B = seq_len(n_smp)) %>%
    ggplot(aes(A, B, z = A + n_fct*B)) +
    stat_summaries_hex(
        aes(alpha = stat(A), fill = stat(B)),
        bins = n_bins,
        funs = list(
            A = function(z) mean(z %% n_fct),
            B = function(z) mean(z/n_fct)
        )
    ) +
    scale_fill_gradient(low  = pal[[1]], high = pal[[2]]) +
    scale_x_continuous(expand = expansion(mult = .5/n_bins)) +
    scale_y_continuous(expand = expansion(mult = 1 /n_bins)) +
    guides(fill = 'none', alpha = 'none') +
    labs(x = expression('#cells' %->% NULL), y = expression('% top rank' %->% NULL)) +
    theme_minimal() +
    theme(panel.grid = element_blank(), axis.text = element_blank())
}
po(2, 2)
plot_gradient_legend()

In [ ]:
po(h <- 1.8, w <- 6.4, p <- 10)
g <- plot_grid(
    plot_gene_relevance_rank(gr, genes = 'Klf4') +
        ggthemes::geom_rangeframe() +
        coord_fixed(.45) +
        ggtitle(NULL) +
        guides(fill = 'none', alpha = 'none') +
        theme(panel.grid = element_blank(), legend.box = 'horizontal'),
    plot_gradient_legend() +
        coord_fixed(),
    plot_gene_relevance(gr) +
        coord_fixed(.45) +
        ggtitle(NULL) +
        guides(colour = guide_legend(title = NULL, keyheight = .2)) +
        theme(panel.grid = element_blank()),
    nrow = 1,
    labels = c('a', '', 'b'),
    rel_widths = c(1, .5, 1.5)
)
g
ggsave('gene-relevance.pdf', g, device = cairo_pdf, height = h, width = w, family = 'TeX Gyre Pagella', pointsize = p)
ggsave('gene-relevance.pdf', g, device = cairo_pdf, height = h, width = w, family = 'Palatino Linotype', pointsize = p)